# FortiOS 8 KVM Lab - Interactive Terminal Access
## Full terminal control directly in Colab

Download lab files and run terminal commands interactively.

---

## STEP 1: Setup & Download Files

In [ ]:
import os
import sys
import subprocess
import urllib.request
from pathlib import Path

print("🚀 Setting up lab environment...\n")

# Create and navigate to lab directory
LAB_DIR = Path('/content/fortios_lab')
LAB_DIR.mkdir(exist_ok=True)
os.chdir(LAB_DIR)

print(f"📁 Working directory: {os.getcwd()}\n")

# Install dependencies
print("[*] Installing Python dependencies...")
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'requests', 'paramiko', 'cryptography', 'flask', 'flask-cors'],
               check=False)
print("✅ Dependencies installed\n")

# Download files
print("[*] Downloading lab files...\n")
files = [
    'kvm_lab_server.py',
    'kvm_lab_cli.py',
    'ui.html',
    'kvm_lab.sh',
    'requirements.txt'
]

base = "https://raw.githubusercontent.com/netanelcyber/HAMIVTZAR/claude/cve-2026-24858-fortios-sso-p1lrtu/cve-research/CVE-SUSPECTED-FORTIOS8-SSO/"

for filename in files:
    try:
        url = base + filename
        print(f"    {filename:30}", end=' ')
        urllib.request.urlretrieve(url, filename)
        size = os.path.getsize(filename) / 1024
        print(f"✓ ({size:.1f} KB)")
    except Exception as e:
        print(f"✗ Error: {e}")

# Make scripts executable
os.chmod('kvm_lab_cli.py', 0o755)
os.chmod('kvm_lab.sh', 0o755)

print(f"\n✅ Setup complete!")
print(f"\n📋 Files ready in: {os.getcwd()}")

## STEP 2: View Lab Files

In [ ]:
import os

print("📂 Lab Files Available:\n")
for filename in sorted(os.listdir('.')):
    if os.path.isfile(filename) and not filename.startswith('.'):
        size = os.path.getsize(filename) / 1024
        status = "🐍" if filename.endswith('.py') else "📄" if filename.endswith('.sh') else "🌐" if filename.endswith('.html') else "📋"
        print(f"  {status} {filename:30} {size:8.1f} KB")

print("\n✅ All files loaded and ready")

## STEP 3: Start Lab Service

In [ ]:
import subprocess
import time
import os

print("🚀 Starting lab HTTP service...\n")

# Start service in background
service_proc = subprocess.Popen(
    ['python3', 'kvm_lab_server.py'],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    cwd=os.getcwd()
)

# Wait for startup
time.sleep(3)

# Check if running
if service_proc.poll() is None:
    print("✅ Service started successfully")
    print(f"   PID: {service_proc.pid}")
    print(f"   URL: http://localhost:5000")
    print(f"   API: http://localhost:5000/api/*")
else:
    print("❌ Service failed to start")
    stdout, stderr = service_proc.communicate()
    print(f"Error: {stderr.decode()}")

# Store for later use
%store service_proc

## STEP 4: Terminal - Run CLI Commands

In [ ]:
# Check service health
!python3 kvm_lab_cli.py health

In [ ]:
# Get service configuration
!python3 kvm_lab_cli.py config

In [ ]:
# Get current API key
import requests
try:
    response = requests.get('http://localhost:5000/api/key')
    api_key = response.json().get('api_key', 'Not set')
    print(f"🔐 Auto-Generated API Key: {api_key}")
except Exception as e:
    print(f"❌ Error retrieving API key: {e}")

In [ ]:
# View lab logs
!python3 kvm_lab_cli.py logs --limit 50

## STEP 5: Terminal - Run Shell Script Commands

In [ ]:
# Get shell script help
!bash kvm_lab.sh help

In [ ]:
# Check health via shell
!bash kvm_lab.sh health

In [ ]:
# View logs via shell
!bash kvm_lab.sh logs 30

## STEP 6: Terminal - Direct Bash Access

In [ ]:
# List files
!ls -lh *.py *.sh *.html 2>/dev/null | tail -10

In [ ]:
# Check Python version
!python3 --version

# Check installed packages
!pip show requests paramiko cryptography | grep -E "Name|Version"

In [ ]:
# Test API endpoint with curl
!curl -s http://localhost:5000/health | python3 -m json.tool

In [ ]:
# View running processes
!ps aux | grep -E "kvm_lab|python3" | grep -v grep

## STEP 7: Python API Client

In [ ]:
import requests
import json

BASE_URL = 'http://localhost:5000'

# Get API key
print("🔐 Retrieving auto-generated API key...\n")
try:
    response = requests.get(f'{BASE_URL}/api/key')
    api_key = response.json().get('api_key')
    print(f"API Key: {api_key}")
    print(f"\n✅ Use this key for authenticated requests")
except Exception as e:
    print(f"Error: {e}")

In [ ]:
# Get lab state
print("📊 Current Lab State:\n")
response = requests.get(f'{BASE_URL}/api/state')
state = response.json().get('state', {})

for key, value in state.items():
    print(f"  {key:20} = {value}")

In [ ]:
# Get logs via Python
print("📋 Recent Lab Logs:\n")
response = requests.get(f'{BASE_URL}/api/logs?limit=20')
logs = response.json().get('logs', [])

print(f"Showing {len(logs)} of {response.json().get('total', 0)} entries\n")

for entry in logs:
    timestamp = entry.get('timestamp', '')[:19]
    level = entry.get('level', 'INFO')
    message = entry.get('message', '')
    icon = {'INFO': '✓', 'ERROR': '✗', 'DEBUG': '→', 'WARN': '⚠'}.get(level, '•')
    print(f"{timestamp} {icon} {level:6} {message}")

## STEP 8: Cleanup

In [ ]:
# Stop service
print("🛑 Stopping lab service...\n")

try:
    %recall service_proc
    if service_proc.poll() is None:
        service_proc.terminate()
        service_proc.wait(timeout=5)
        print("✅ Service stopped")
    else:
        print("⚠️  Service already stopped")
except:
    # Fallback to pkill
    !pkill -f kvm_lab_server || true
    print("✅ Service stopped")

print("\n✅ Cleanup complete")